<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task3/Task3_Model1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#Task 3
## Algorithm Selection Justification

REGRESSION MODELS — Predict exact ACTUAL_COST (£):

1. Linear Regression (Baseline)
   Selected as the interpretable baseline model. NHS prescription
   costs often scale linearly with quantity and drug type. This
   model establishes a performance floor — all other models must
   outperform it to justify their complexity.

2. Random Forest Regressor
   Selected for its ability to capture complex non-linear
   interactions between 15 features across 18M records. NHS
   prescribing patterns vary significantly by region, GP practice
   and drug category — interactions that linear models miss entirely.

CLASSIFICATION MODELS — Predict HIGH_COST flag (£50 threshold):

3. Logistic Regression (Baseline Classifier)
   Selected as the probabilistic baseline classifier. Outputs a
   probability score (0-1) indicating likelihood of high cost —
   directly usable in NHS budget monitoring workflows. Simple,
   fast and highly interpretable.

4. Decision Tree Classifier
   Selected for its unmatched explainability. Produces human-readable
   rules that NHS managers, GPs and policy makers can understand and
   act on without data science expertise. Critical for NHS trust and
   adoption of ML systems.

WHY NOT OTHERS:
- SVM: Does not scale to 18M rows efficiently in PySpark
- KNN: O(n²) complexity — computationally impossible at this scale  
- K-Means: Clustering only — cannot predict cost values
- Deep Learning: GPU required, overkill for structured tabular data

In [ ]:
#Task 3 — ML Model Portfolio

#import and start pyspark
!pip install pyspark -q

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [ ]:
#load data from task 2
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# Load preprocessed train/test data saved in Task 2
train_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet'
)
test_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet'
)

print(f"Training rows: {train_df.count():,}")
print(f"Testing rows: {test_df.count():,}")

# Use a smaller sample for model tuning - due to RAM issue
train_sample = train_df.sample(0.03, seed=42)
test_sample = test_df.sample(0.03, seed=42)

print(f"Training sample rows: {train_sample.count():,}")
print(f"Testing sample rows: {test_sample.count():,}")

print("Data loaded successfully!")

Mounted at /content/drive
Training rows: 1,469,696
Testing rows: 367,299
Training sample rows: 44,478
Testing sample rows: 11,072
Data loaded successfully!


In [ ]:
# Check feature vector dimensions

first_row = train_sample.select("scaled_features").first()

print("Number of features:")
print(len(first_row["scaled_features"]))

Number of features:
87


In [ ]:
#Model 1: Linear Regression -
#predicts ACTUAL_COST

#IMPORT Linear Regression
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator #automatically tests all settings and picks the best

# ── MODEL 1: LINEAR REGRESSION ──
# predicts exact ACTUAL_COST (£)
# WHY: Establishes performance baseline
# If complex models don't beat this simple model, something is fundamentally wrong with the data or pipeline

# Define the model
lr_Model = LinearRegression(
    featuresCol='scaled_features',  #column containing all our prepared features
    labelCol='ACTUAL_COST'  #column we want to predict
)

print("Linear Regression model defined!")
print(f"Feature column: scaled_features")
print(f"Target column: ACTUAL_COST")

Linear Regression model defined!
Feature column: scaled_features
Target column: ACTUAL_COST


In [12]:
# HYPERPARAMETER TUNING ──
# CrossValidator tests different parameter combinations
# and selects the one that produces the lowest prediction error.

# regParam - controls the amount of regularisation:
# lower values allow the model to fit more closely to the data,
# while higher values help reduce overfitting.

# elasticNetParam controls the regularisation type:
# 0.0 uses Ridge regression (L2)
# 0.5 uses a combination of Ridge and Lasso (L1 + L2)


#create a grid of all setting combinations to test
lr_grid = (
    ParamGridBuilder()
    .addGrid(lr_Model.regParam, [0.01])
    .addGrid(lr_Model.elasticNetParam, [0.0])
    .build()
)

lr_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',  #REAL value
    predictionCol='prediction',  #model's guess
    metricName='rmse' #Root Mean Squared Error - to measure model performance.
)
# A lower RMSE indicates that predictions are closer to the actual values.

# Cross-validation helps provide a more reliable estimate
# of model performance by training and testing on different
# portions of the dataset.

# numFold = 3 means:
# Two parts are used for training
# One part is used for validation
# The process repeats three times

lr_cv = CrossValidator(
    estimator=lr_Model,
    estimatorParamMaps=lr_grid,
    evaluator=lr_evaluator,
    numFolds=2,      # fewer folds = less memory usage
    seed=42, # ensures results can be reproduced
    parallelism=1    # train one model at a time
)

print("CrossValidator configured!")
print(f"Parameter combinations: {len(lr_grid)}")
print(f"Total training runs: {len(lr_grid) * 2}")  #2 folds
print("Parameters being tuned:")
print("  regParam: [0.01]")
print("  elasticNetParam: [0.0]")

CrossValidator configured!
Parameter combinations: 1
Total training runs: 2
Parameters being tuned:
  regParam: [0.01]
  elasticNetParam: [0.0]


In [13]:
# TRAINING
# This runs all 12 combinations and picks the best
print("Training Linear Regression with CrossValidator")

lr_start = time.time()           # record start time
lr_model = lr_cv.fit(train_sample)  # train all 12 combinations
lr_end = time.time()             # record end time
lr_time = round(lr_end - lr_start, 2)  # calculate seconds

print(f"\nTime Taken for Training: {lr_time} seconds!")

Training Linear Regression with CrossValidator

Time Taken for Training: 42.91 seconds!


In [14]:
#RESULTS
# BEST HYPERPARAMETERS
# Extract which settings CrossValidator selected as best
best_lr = lr_model.bestModel
print(f"Best regParam: {best_lr._java_obj.getRegParam()}")
print(f"Best elasticNetParam: {best_lr._java_obj.getElasticNetParam()}")

# EVALUATION
# Apply best model to test data (data it has never seen)
lr_predictions = lr_model.transform(test_sample)

# RMSE — Root Mean Square Error (average £ error)
lr_rmse = lr_evaluator.evaluate(lr_predictions)

# R² — how much of cost variation the model explains
# 1.0 = perfect, 0.0 = no better than guessing average
lr_r2_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',
    predictionCol='prediction',
    metricName='r2'
)
lr_r2 = lr_r2_evaluator.evaluate(lr_predictions)

# MAE — Mean Absolute Error (average £ difference)
lr_mae_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',
    predictionCol='prediction',
    metricName='mae'
)
lr_mae = lr_mae_evaluator.evaluate(lr_predictions)

# ── RESULTS ──
print("═" * 45)
print("   LINEAR REGRESSION — FINAL RESULTS")
print("═" * 45)
print(f"   RMSE:          £{lr_rmse:.4f}")
print(f"   R²:             {lr_r2:.4f}")
print(f"   MAE:           £{lr_mae:.4f}")
print(f"   Training Time:  {lr_time} seconds")
print("═" * 45)

# Show sample predictions vs actual
print("\nSample Predictions vs Actual:")
lr_predictions.select(
    'ACTUAL_COST',
    'prediction'
).show(10)

Best regParam: 0.01
Best elasticNetParam: 0.0
═════════════════════════════════════════════
   LINEAR REGRESSION — FINAL RESULTS
═════════════════════════════════════════════
   RMSE:          £8.2251
   R²:             0.9988
   MAE:           £2.4268
   Training Time:  42.91 seconds
═════════════════════════════════════════════

Sample Predictions vs Actual:
+-----------+------------------+
|ACTUAL_COST|        prediction|
+-----------+------------------+
|   47.19265|49.095658830451576|
|     7.5424| 5.870309929999193|
|    23.5696|20.379763354264746|
|    5.90333| 6.326085745779122|
|    2.97687|2.4313588954368455|
|    5.33025|5.0048025735312045|
|   21.29619|20.480085859729364|
|    3.35923|2.0581338795403967|
|   13.85592|14.298151025201548|
|    61.5124| 58.88072006810211|
+-----------+------------------+
only showing top 10 rows
